# 08 · Önceden kayıtlı test: The Strat (sade)
## Bar tipleri 1 / 2u / 2d / 3 · 2-1-2, 3-1-2, 2-2 dönüş · FTFC · günlük ve haftalık · 2010–2026

Bu not defteri **KARARLAR.md → "Önceden kayıtlı test: The Strat, sade (08)"** bölümündeki kuralları **birebir** uygular. Kurallar 17/11/2026'da, strateji gerçek veride hiç çalıştırılmadan önce kayda geçirildi. Broadening formation ve magnitude seviyeleri bilinçli olarak dışarıda.

**Düzeltme (17/11/2026, ilk çalıştırmadan sonra):** hisse başı komisyon (0,0035 $) ilk sürümde bölünmeye göre düzeltilmiş fiyata uygulanıyordu; NVDA gibi çok bölünen hisselerde eski yıllarda maliyet onlarca kat şişiyordu. Komisyon artık bölünme çarpanıyla gerçek hisse fiyatına göre alınır. Kurallar ve eşikler değişmedi; ilk sürümün çıktısı KARARLAR.md'de ayrıca kayıtlı.

**Akış:**
1. yfinance günlük veri (bölünme ve temettü düzeltmeli), haftalık barlar günlükten üretilir.
2. Her bar bir öncekine göre sınıflanır: **1** inside, **2u / 2d** tek uç aşan, **3** outside (eşit uç "aşmadı" sayılır).
3. Kurulumlar: **2-1-2 devam / dönüş**, **3-1-2**, **2-2 dönüş**. Giriş son barın ucunun kırılması, stop öbür ucu, hedef bir önceki barın ucu.
4. **FTFC:** giriş fiyatı üst zaman dilimlerinin açılışının doğru tarafında mı (günlükte ay + hafta, haftalıkta çeyrek + ay).
5. Karıştırılmış bar kıyası, önceden kayıtlı test, sonra karar vermeyen raporlar.

**Ölçüt hedefe ulaşma oranı değil, maliyet sonrası ortalama R'dır.** Sahte veri denemesinde inside bar kurulumları rastgele piyasada bile hedefe %85–100 ulaştı ama ilk hedef ~0,3R olduğu için ortalama R negatifti.

*Öğrenme amaçlıdır, yatırım tavsiyesi değildir.*

In [ ]:
#@title ⚙️ Ayarlar
PERMUTASYON_TEKRAR = 5000 #@param {type:"integer"}
KARISTIRMA_TEKRAR = 400 #@param {type:"integer"}
#@markdown ---
#@markdown **Aşağıdakiler KARARLAR.md'de sabittir — değiştirme.**
VERI_BASLANGIC = "2010-01-01"
VERI_BITIS = "2026-11-14"        # yfinance'te bitiş günü hariç: son gün 13/11/2026
ORNEKLEM_DISI = "2023-01-01"     # 2010–2022 örneklem içi, 2023–13/11/2026 örneklem dışı
ETFLER = ["SPY", "QQQ", "IWM"]
MAG7 = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA"]
KOMISYON_HISSE = 0.0035          # $ / hisse, her yön
KAYMA_BPS = {"ETF": 1, "Mag7": 3}  # her yön
MAKS_BAR = {"D": 10, "W": 4}     # zaman stopu
ALFA = 0.025 / 2                 # iki zaman dilimi (D, W) için Bonferroni
ASGARI_ISLEM = {"D": 200, "W": 100}
TOHUM = 2026

## 1 · Kurulum ve veri

In [ ]:
import subprocess, sys
try:
    import yfinance as yf
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=True)
    import yfinance as yf
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

def tr_sayi(x, ond=3):
    if x is None or (isinstance(x, (float, np.floating)) and np.isnan(x)):
        return "–"
    s = f"{abs(x):,.{ond}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return ("-" if x < 0 else "") + s

def tr_yuzde(x, ond=1):
    return "–" if x is None or pd.isna(x) else "%" + tr_sayi(100 * x, ond)

def veri_indir(sembol, bitis=VERI_BITIS):
    df = yf.download(sembol, start=VERI_BASLANGIC, end=bitis, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[["Open", "High", "Low", "Close"]].dropna()
    df.index = pd.to_datetime(df.index).tz_localize(None)
    return df

def bolunme_carpani(sembol, index):
    # Düzeltilmiş fiyat × çarpan ≈ o günkü gerçek hisse fiyatı. Hisse başı komisyon gerçek fiyata göre alınır
    f = pd.Series(1.0, index=index)
    for tarih, oran in yf.Ticker(sembol).splits.items():
        if oran and oran > 0:
            f[index < pd.Timestamp(tarih).tz_localize(None)] *= oran
    return f

GRUP = {s: "ETF" for s in ETFLER} | {s: "Mag7" for s in MAG7}
VERI, CARPAN = {}, {}
for s in ETFLER + MAG7:
    VERI[s] = veri_indir(s)
    CARPAN[s] = bolunme_carpani(s, VERI[s].index)
    print(f"{s:6s} {len(VERI[s]):5d} gün  {VERI[s].index[0]:%d/%m/%Y} – {VERI[s].index[-1]:%d/%m/%Y}  "
          f"bölünme çarpanı (ilk gün) {tr_sayi(CARPAN[s].iloc[0], 0)}")

## 2 · Bar tipleri, kurulumlar, simülasyon

In [ ]:
def haftalik(gunluk):
    g = gunluk.copy()
    g["ilk_gun"] = g.index
    w = g.resample("W-FRI").agg({"Open": "first", "High": "max", "Low": "min", "Close": "last", "ilk_gun": "first"})
    return w.dropna()

def bar_tipi(df):
    ust = df["High"] > df["High"].shift(1)
    alt = df["Low"] < df["Low"].shift(1)
    t = pd.Series(np.select([ust & alt, ust, alt], ["3", "2u", "2d"], default="1"), index=df.index, dtype=object)
    t.iloc[0] = None
    return t

def kurulumlari_bul(df):
    # Bar i tamamlandığında bar i+1'de tetiklenebilecek kurulumlar
    tip = bar_tipi(df).values
    H, L = df["High"].values, df["Low"].values
    k = []
    for i in range(2, len(df)):
        if tip[i] == "1" and tip[i - 1] in ("2u", "2d", "3"):
            for yon in ("long", "short"):
                giris, stop = (H[i], L[i]) if yon == "long" else (L[i], H[i])
                hedef = H[i - 1] if yon == "long" else L[i - 1]
                if (yon == "long" and hedef <= giris) or (yon == "short" and hedef >= giris):
                    continue
                if tip[i - 1] == "3":
                    ad = "3-1-2"
                else:
                    ad = "2-1-2 devam" if (tip[i - 1] == "2u") == (yon == "long") else "2-1-2 dönüş"
                k.append((i, ad, yon, giris, stop, hedef))
        if tip[i] == "2d" and H[i - 1] > H[i]:
            k.append((i, "2-2 dönüş", "long", H[i], L[i], H[i - 1]))
        if tip[i] == "2u" and L[i - 1] < L[i]:
            k.append((i, "2-2 dönüş", "short", L[i], H[i], L[i - 1]))
    return pd.DataFrame(k, columns=["i", "kurulum", "yon", "giris", "stop", "hedef"])

def ftfc_acilislari(gunluk, df, tf):
    # Tetik anında bilinen üst zaman dilimi açılışları (ileriye bakış yok)
    if tf == "D":
        ay = gunluk.groupby(gunluk.index.to_period("M"))["Open"].transform("first")
        hf = gunluk.groupby(gunluk.index.to_period("W-FRI"))["Open"].transform("first")
        return np.column_stack([ay.values, hf.values])
    ay = gunluk.groupby(gunluk.index.to_period("M"))["Open"].first()
    cy = gunluk.groupby(gunluk.index.to_period("Q"))["Open"].first()
    ilk = pd.DatetimeIndex(df["ilk_gun"])          # haftanın ilk işlem gününün ayı / çeyreği
    return np.column_stack([ay.reindex(ilk.to_period("M")).values, cy.reindex(ilk.to_period("Q")).values])

def cikis_bul(O, H, L, C, j, s, uzun, maks, iyimser):
    son = min(j + maks, len(C)) - 1
    for m in range(j, son + 1):
        if m > j:
            if (uzun and O[m] <= s.stop) or (not uzun and O[m] >= s.stop):
                return O[m], "stop (boşluk)"
            if (uzun and O[m] >= s.hedef) or (not uzun and O[m] <= s.hedef):
                return O[m], "hedef (boşluk)"
        stop_v = (L[m] <= s.stop) if uzun else (H[m] >= s.stop)
        if m == j and iyimser:
            stop_v = False               # iyimser: giriş barında önce stop, sonra giriş varsayılır
        hedef_v = (H[m] >= s.hedef) if uzun else (L[m] <= s.hedef)
        if stop_v and hedef_v:
            return (s.hedef, "hedef") if iyimser else (s.stop, "stop")
        if stop_v:
            return s.stop, "stop"
        if hedef_v:
            return s.hedef, "hedef"
    return C[son], "zaman"

def simule_et(gunluk, sembol, tf):
    df = gunluk if tf == "D" else haftalik(gunluk)
    O, H, L, C = (df[c].values for c in ("Open", "High", "Low", "Close"))
    acilis = ftfc_acilislari(gunluk, df, tf)
    kayma = KAYMA_BPS[GRUP.get(sembol, "Mag7")] / 1e4
    carpan = CARPAN[sembol].reindex(df.index, method="ffill").fillna(1.0).values
    sonuc = []
    for s in kurulumlari_bul(df).itertuples():
        j = s.i + 1
        if j >= len(df):
            continue
        uzun = s.yon == "long"
        if not (H[j] >= s.giris if uzun else L[j] <= s.giris):
            continue
        if s.kurulum != "2-2 dönüş" and (L[j] <= s.stop if uzun else H[j] >= s.stop):
            continue                     # inside barın iki ucu da aynı barda kırıldı: sıra bilinmez
        giris = max(O[j], s.giris) if uzun else min(O[j], s.giris)
        risk = abs(giris - s.stop)
        if risk <= 0 or (uzun and giris >= s.hedef) or (not uzun and giris <= s.hedef):
            continue                     # boşlukla stopun ya da hedefin ötesinde açıldı
        ac = acilis[j]
        ftfc = None if np.isnan(ac).any() else bool(np.all(giris > ac) if uzun else np.all(giris < ac))
        maliyet = 2 * kayma * giris + 2 * KOMISYON_HISSE / carpan[j]   # komisyon gerçek hisse sayısına göre
        satir = dict(sembol=sembol, grup=GRUP.get(sembol, "Mag7"), tf=tf, tarih=df.index[j], kurulum=s.kurulum,
                     yon=s.yon, ftfc=ftfc, giris=giris, stop=s.stop, hedef=s.hedef,
                     hedef_R=abs(s.hedef - giris) / risk, risk_bps=1e4 * risk / giris)
        for ad, iy in (("k", False), ("i", True)):
            cik, neden = cikis_bul(O, H, L, C, j, s, uzun, MAKS_BAR[tf], iy)
            brut = (cik - giris) if uzun else (giris - cik)
            satir[f"R_{ad}"] = (brut - maliyet) / risk
            satir[f"neden_{ad}"] = neden
        sonuc.append(satir)
    return pd.DataFrame(sonuc)

ISLEM = pd.concat([simule_et(VERI[s], s, tf) for s in VERI for tf in ("D", "W")], ignore_index=True)
ISLEM["donem"] = np.where(ISLEM.tarih < pd.Timestamp(ORNEKLEM_DISI), "içi", "dışı")
print(ISLEM.groupby(["tf", "ftfc"], dropna=False).size())

## 3 · Karıştırılmış bar kıyası (2. şartın sıfır dağılımı)
Her sembolün günlük barları (gap, yükseklik, düşüklük ve kapanış oranları) rastgele sıraya dizilip fiyat yolu yeniden kurulur. Bar şekilleri, oynaklık ve ortalama eğilim aynı kalır; barların sırası, yani Strat'ın okuduğu yapı, yok olur. Aynı kurallar bu sentetik veride çalıştırılır.

Neden gerekli: FTFC uyumlu kurulumların hedefi mekanik olarak daha yakın çıkıyor ve yükselen piyasada uyumlu long'lar eğilimden pay alıyor. Bu yüzden sırası tamamen rastgele bir piyasada bile FTFC farkı sıfır olmuyor. Önceden kayıtlı testin 2. şartı farkı sıfırla değil, bu dağılımla karşılaştırır. Bu bölüm Colab'da birkaç dakika sürer.

In [ ]:
def karistir(g, rng):
    Cp = g["Close"].shift(1)
    oran = pd.DataFrame({"gap": g["Open"] / Cp, "h": g["High"] / g["Open"], "l": g["Low"] / g["Open"],
                         "c": g["Close"] / g["Open"]}).iloc[1:]
    oran = oran.iloc[rng.permutation(len(oran))].values
    O = g["Close"].iloc[0] * np.cumprod(oran[:, 0] * oran[:, 3]) / oran[:, 3]
    yeni = pd.DataFrame({"Open": O, "High": O * oran[:, 1], "Low": O * oran[:, 2], "Close": O * oran[:, 3]},
                        index=g.index[1:])
    return yeni

rng_k = np.random.default_rng(TOHUM + 1)
ozetler = []
for r in range(KARISTIRMA_TEKRAR):
    t = pd.concat([simule_et(karistir(VERI[s], rng_k), s, tf) for s in VERI for tf in ("D", "W")], ignore_index=True)
    m = t.groupby(["tf", "ftfc"])["R_k"].mean()
    ozetler.append({tf: (m.get((tf, True), np.nan), m.get((tf, False), np.nan)) for tf in ("D", "W")})
    if (r + 1) % 50 == 0:
        print(f"{r + 1}/{KARISTIRMA_TEKRAR} tekrar")
KIYAS_FARKLAR = {tf: np.array([o[tf][0] - o[tf][1] for o in ozetler]) for tf in ("D", "W")}
satir = []
for tf in ("D", "W"):
    for e, ad in ((True, "FTFC uyumlu"), (False, "uyumsuz")):
        g = ISLEM.loc[(ISLEM.tf == tf) & (ISLEM.ftfc == e), "R_k"].mean()
        k = np.nanmean([o[tf][0 if e else 1] for o in ozetler])
        satir.append(dict(tf=tf, grup=ad, gercek=g, karistirilmis=k, sira_katkisi=g - k))
display(pd.DataFrame(satir).style.format({c: tr_sayi for c in ("gercek", "karistirilmis", "sira_katkisi")}))
for tf in ("D", "W"):
    print(f"{tf}: karıştırılmış veride FTFC farkı ortalaması {tr_sayi(np.nanmean(KIYAS_FARKLAR[tf]))} R, "
          f"%95'lik dilim {tr_sayi(np.nanpercentile(KIYAS_FARKLAR[tf], 95))} R")

## 4 · Önceden kayıtlı test

In [ ]:
RNG = np.random.default_rng(TOHUM)

def fark(d, kol):
    return d.loc[d.ftfc == True, kol].mean() - d.loc[d.ftfc == False, kol].mean()

def isaret_cevirme_p(d, kol="R_k"):
    # H0: ortalama net R = 0. Gün bazında işaret çevirme (aynı günün işlemleri birlikte)
    gun = d.groupby("tarih")[kol].sum().values
    n = len(d); gozlem = gun.sum() / n
    perm = np.empty(PERMUTASYON_TEKRAR)
    for b in range(PERMUTASYON_TEKRAR):
        perm[b] = (gun * RNG.choice((-1.0, 1.0), size=len(gun))).sum() / n
    return gozlem, (1 + (perm >= gozlem).sum()) / (1 + PERMUTASYON_TEKRAR)

def fark_p(d, tf, kol="R_k"):
    # H0: barların sırası bilgi taşımıyor. Gerçek fark, karıştırılmış barlardaki farklarla karşılaştırılır
    gozlem = fark(d, kol)
    k = KIYAS_FARKLAR[tf][~np.isnan(KIYAS_FARKLAR[tf])]
    return gozlem, (1 + (k >= gozlem).sum()) / (1 + len(k))

SONUC_SATIRLARI = []
for tf in ("D", "W"):
    d = ISLEM[(ISLEM.tf == tf) & ISLEM.ftfc.notna()].copy()
    d["ftfc"] = d.ftfc.astype(bool)
    uy = d[d.ftfc]
    ort, p1 = isaret_cevirme_p(uy)
    frk, p2 = fark_p(d, tf)
    ici, disi = d[d.donem == "içi"], d[d.donem == "dışı"]
    ort_ici, ort_disi = ici.loc[ici.ftfc, "R_k"].mean(), disi.loc[disi.ftfc, "R_k"].mean()
    frk_ici, frk_disi = fark(ici, "R_k"), fark(disi, "R_k")
    ort_iy, frk_iy = uy["R_i"].mean(), fark(d, "R_i")
    n_uy, n_un = int(d.ftfc.sum()), int((~d.ftfc).sum())
    sartlar = {
        "1 · FTFC uyumlu ort. net R > 0, p < α": ort > 0 and p1 < ALFA,
        "2 · fark (uyumlu − uyumsuz) > 0, karıştırılmış barlara göre p < α": frk > 0 and p2 < ALFA,
        "3 · ortalama ve fark örneklem içi ve dışında ayrı ayrı > 0": min(ort_ici, ort_disi, frk_ici, frk_disi) > 0,
        "4 · ortalama ve fark iyimser varsayımda da > 0": ort_iy > 0 and frk_iy > 0,
        f"5 · her grupta en az {ASGARI_ISLEM[tf]} işlem": min(n_uy, n_un) >= ASGARI_ISLEM[tf],
    }
    karar = "DOĞRULANDI" if all(sartlar.values()) else "DOĞRULANMADI"
    display(Markdown(f"### {'Günlük' if tf == 'D' else 'Haftalık'} ({tf}) · **{karar}**  (α = {tr_sayi(ALFA, 4)})"))
    display(pd.DataFrame({"şart": list(sartlar), "sağlandı mı": ["evet" if v else "hayır" for v in sartlar.values()]}))
    SONUC_SATIRLARI.append(
        f"- {tf}: {karar} · FTFC uyumlu net R {tr_sayi(ort)}, p={tr_sayi(p1, 4)} · fark {tr_sayi(frk)} R, p={tr_sayi(p2, 4)}"
        f" · örneklem içi {tr_sayi(ort_ici)} / {tr_sayi(frk_ici)}, dışı {tr_sayi(ort_disi)} / {tr_sayi(frk_disi)}"
        f" · iyimser {tr_sayi(ort_iy)} / {tr_sayi(frk_iy)} · karıştırılmış fark ort. {tr_sayi(np.nanmean(KIYAS_FARKLAR[tf]))} · işlem {tr_sayi(n_uy, 0)} / {tr_sayi(n_un, 0)}")

display(Markdown("### KARARLAR.md için (değiştirmeden kopyala)"))
print("\n".join(SONUC_SATIRLARI))

## 5 · Raporlanır ama karar vermez

In [ ]:
def tablo(d, gruplar):
    t = d.groupby(gruplar, dropna=False).agg(
        islem=("R_k", "size"),
        hedefe_ulasma=("neden_k", lambda x: x.str.startswith("hedef").mean()),
        ort_R_kotumser=("R_k", "mean"), ort_R_iyimser=("R_i", "mean"),
        medyan_hedef_R=("hedef_R", "median"), medyan_risk_bps=("risk_bps", "median"))
    return t.style.format({"hedefe_ulasma": tr_yuzde, "ort_R_kotumser": tr_sayi, "ort_R_iyimser": tr_sayi,
                           "medyan_hedef_R": lambda x: tr_sayi(x, 2), "medyan_risk_bps": lambda x: tr_sayi(x, 0)})

for baslik, d, g in [
    ("Kurulum ve yön", ISLEM, ["tf", "kurulum", "yon", "ftfc"]),
    ("Yalnızca hedef ≥ 1R", ISLEM[ISLEM.hedef_R >= 1], ["tf", "kurulum", "ftfc"]),
    ("ETF / Mag7", ISLEM, ["tf", "grup", "ftfc"]),
    ("Sembol", ISLEM, ["tf", "sembol", "ftfc"]),
    ("Dönem", ISLEM, ["tf", "donem", "ftfc"]),
]:
    display(Markdown(f"#### {baslik}")); display(tablo(d, g))

yil = ISLEM.assign(yil=ISLEM.tarih.dt.year).pivot_table(index="yil", columns=["tf", "ftfc"], values="R_k", aggfunc="mean")
display(Markdown("#### Yıllara göre ortalama net R (kötümser)")); display(yil.style.format(tr_sayi))

## 6 · Son tamamlanan bardaki kurulumlar (bilgi amaçlı, test dışı)
Bu bölüm testten bağımsızdır: güncel veriyi indirir ve son tamamlanan günlük / haftalık barda tetik bekleyen kurulumları listeler. İsabet kaydı için kullanılacak. ABD seansı kapandıktan sonra (TSİ 24:00 sonrası) çalıştır; seans içinde günlük son bar henüz tamamlanmamıştır. Haftalıkta bitmemiş hafta otomatik olarak dışarıda bırakılır.

In [ ]:
satir = []
for s in ETFLER + MAG7:
    g = veri_indir(s, bitis=None)
    for tf in ("D", "W"):
        df = g if tf == "D" else haftalik(g)
        if tf == "W" and df.index[-1] > g.index[-1]:
            df = df.iloc[:-1]            # bu hafta henüz bitmedi: son tamamlanan hafta
        k = kurulumlari_bul(df)
        son = k[k.i == len(df) - 1]
        ac = ftfc_acilislari(g, df, tf)[-1]
        for x in son.itertuples():
            uzun = x.yon == "long"
            satir.append(dict(sembol=s, tf=tf, bar=f"{df.index[-1]:%d/%m/%Y}", kurulum=x.kurulum, yon=x.yon,
                              tetik=round(x.giris, 2), stop=round(x.stop, 2), hedef=round(x.hedef, 2),
                              hedef_R=round(abs(x.hedef - x.giris) / abs(x.giris - x.stop), 2),
                              ftfc_simdi=bool(np.all(x.giris > ac) if uzun else np.all(x.giris < ac))))
display(pd.DataFrame(satir) if satir else Markdown("Son barda kurulum yok."))